# 03_silver_tests — Compare Bronze and Silver

Run this notebook after `02_silver`. It checks that Silver is complete, correct, and consistent with Bronze, and produces a PASS/FAIL report.

| Test group | What it proves |
|---|---|
| Completeness | No record was lost or invented between Bronze and Silver |
| Versions | Duplicates were resolved and the latest version was kept |
| Data types | Every column has the expected type |
| Value tracing | Every cleaned amount comes from the Bronze value in the same file |
| Data quality rules | VALID rows really meet every rule; INVALID rows have a reason |
| Known scenarios | Each messy value from the Day 2 files was handled as designed |
| Expected counts | Row, valid, and invalid counts match the lab data |

The last cell stops with an error if any test fails, so a scheduled job that runs this notebook is marked as failed.

## Step 1: Setup

**Why:** `record` stores each test result. `norm_key` cleans a raw Bronze key the same way Silver should (trim, single spaces, upper case), so Bronze and Silver keys can be compared. `PARTNERS` lists, for each partner, the tables, the key column, the amount and date columns, and the expected data types.

In [0]:
from pyspark.sql import functions as F

results = []

def record(partner, test, expected, actual):
    """Store one test result. PASS when expected equals actual."""
    status = "PASS" if str(expected) == str(actual) else "FAIL"
    results.append((partner, test, str(expected), str(actual), status))

def norm_key(col_name):
    """Normalise a raw Bronze key the same way Silver should: trim, single spaces, upper case."""
    return F.upper(F.regexp_replace(F.trim(F.col(col_name)), r"\s+", " "))

def bronze_file_date():
    return F.expr("to_date(try_to_timestamp(regexp_extract(_source_file, '([0-9]{8})', 1), 'yyyyMMdd'))")

def bronze_amount(col_name):
    return F.expr(f"try_cast(regexp_replace({col_name}, '[$, ]', '') AS DECIMAL(12,2))")

PARTNERS = {
    "BANK": {
        "bronze": "insurance.bronze.bank_settlement",
        "silver": "insurance.silver.bank_settlement",
        "key": "bank_ref",
        "amounts": ["amount"],
        "dates": ["settlement_date"],
        "types": {"bank_ref": "string", "payment_id": "int", "claim_id": "int",
                  "amount": "decimal(12,2)", "settlement_date": "date", "file_date": "date"},
    },
    "TPA": {
        "bronze": "insurance.bronze.tpa_settlement",
        "silver": "insurance.silver.tpa_settlement",
        "key": "tpa_ref",
        "amounts": ["billed_amount", "approved_amount"],
        "dates": ["service_date"],
        "types": {"tpa_ref": "string", "claim_id": "int", "billed_amount": "decimal(12,2)",
                  "approved_amount": "decimal(12,2)", "service_date": "date", "file_date": "date"},
    },
    "REPAIR": {
        "bronze": "insurance.bronze.repair_invoice",
        "silver": "insurance.silver.repair_invoice",
        "key": "invoice_no",
        "amounts": ["labor_amount", "total_amount"],
        "dates": ["invoice_date"],
        "types": {"invoice_no": "string", "claim_id": "int", "labor_amount": "decimal(12,2)",
                  "parts_amount": "decimal(12,2)", "total_amount": "decimal(12,2)",
                  "invoice_date": "date", "file_date": "date"},
    },
}

## Step 2: Completeness

**Why:** proves that no record was lost and none was invented.
- Silver must have exactly one row for every distinct key in Bronze.
- Every Bronze key must exist in Silver, and every Silver key must exist in Bronze (`left_anti` join returns the rows with no match).
- Silver must have no duplicate keys and no null keys.

In [0]:
for name, p in PARTNERS.items():
    k = p["key"]
    b = spark.table(p["bronze"]).withColumn("_key", norm_key(k))
    s = spark.table(p["silver"])
    bronze_keys = b.select("_key").distinct()
    silver_keys = s.select(F.col(k).alias("_key"))

    record(name, "Bronze rows (information)", b.count(), b.count())
    record(name, "Silver rows = distinct keys in Bronze", bronze_keys.count(), s.count())
    record(name, "Every Bronze key is in Silver", 0, bronze_keys.join(silver_keys, "_key", "left_anti").count())
    record(name, "Every Silver key exists in Bronze", 0, silver_keys.join(bronze_keys, "_key", "left_anti").count())
    record(name, "No duplicate keys in Silver", 0, s.groupBy(k).count().filter("count > 1").count())
    record(name, "No null keys in Silver", 0, s.filter(F.col(k).isNull()).count())

## Step 3: Versions

**Why:** Bronze keeps every version of a record; Silver keeps one.
- `record_versions` in Silver must equal the number of Bronze rows for that key.
- The row kept must come from the newest file for that key.

In [0]:
for name, p in PARTNERS.items():
    k = p["key"]
    b = (spark.table(p["bronze"])
         .withColumn("_key", norm_key(k))
         .withColumn("_file_date", bronze_file_date()))
    s = spark.table(p["silver"])
    per_key = b.groupBy("_key").agg(F.count("*").alias("bronze_versions"),
                                    F.max("_file_date").alias("latest_file_date"))
    j = s.join(per_key, s[k] == per_key["_key"])

    record(name, "record_versions = Bronze rows per key", 0,
           j.filter(F.col("record_versions") != F.col("bronze_versions")).count())
    record(name, "Latest file kept for each key", 0,
           j.filter(F.col("file_date") != F.col("latest_file_date")).count())

## Step 4: Data types

**Why:** Gold and reports depend on amounts being decimals, IDs being integers, and dates being dates. A wrong type (for example `decimal(13,2)` instead of `decimal(12,2)`) can break joins and unions later.

In [0]:
for name, p in PARTNERS.items():
    actual = dict(spark.table(p["silver"]).dtypes)
    wrong = [f"{c}: expected {t}, found {actual.get(c)}" for c, t in p["types"].items() if actual.get(c) != t]
    record(name, "Column data types", "none wrong", "none wrong" if not wrong else "; ".join(wrong))

## Step 5: Value tracing

**Why:** proves the cleaning did not change any amount. For every Silver row, the amount is compared with the Bronze value from the same key and the same source file, after removing `$`, commas, and spaces. Every Silver row must have a matching Bronze value.

In [0]:
for name, p in PARTNERS.items():
    k = p["key"]
    b = spark.table(p["bronze"]).withColumn("_key", norm_key(k))
    s = spark.table(p["silver"])
    silver_rows = s.count()
    for a in p["amounts"]:
        bronze_vals = b.select("_key", "_source_file", bronze_amount(a).alias("_bronze_value"))
        traced = (s.select(F.col(k).alias("_key"), "_source_file", F.col(a).alias("_silver_value"))
                   .join(bronze_vals, ["_key", "_source_file"])
                   .filter(F.col("_silver_value").eqNullSafe(F.col("_bronze_value")))
                   .select("_key").distinct().count())
        record(name, f"{a}: every Silver value traces to the Bronze value in the same file", silver_rows, traced)

## Step 6: Data quality rules

**Why:** proves the `dq_status` flag can be trusted.
- `VALID` rows have no errors, and `INVALID` rows have at least one.
- `VALID` rows have every required value, no negative amounts, an allowed status, approved ≤ billed, a service date not after the file date, and total = labor + parts.
- Text is standardised: keys trimmed and upper case, status upper case, providers approved or warned, shop names trimmed.

In [0]:
VALID_BANK_STATUS = ["CLEARED", "RETURNED"]
APPROVED_PROVIDERS = ["Memorial Hermann", "Baylor Scott & White", "HCA Houston",
                      "Methodist Hospital", "CareNow Urgent Care"]

for name, p in PARTNERS.items():
    k = p["key"]
    s = spark.table(p["silver"])
    valid = s.filter(F.col("dq_status") == "VALID")

    record(name, "dq_status matches dq_errors", 0, s.filter(
        ((F.col("dq_status") == "VALID") & (F.size("dq_errors") > 0)) |
        ((F.col("dq_status") == "INVALID") & (F.size("dq_errors") == 0))).count())
    record(name, "Keys are trimmed and upper case", 0,
           s.filter(F.col(k) != F.upper(F.trim(F.col(k)))).count())
    for a in p["amounts"]:
        record(name, f"VALID rows have {a}", 0, valid.filter(F.col(a).isNull()).count())
        record(name, f"VALID rows have no negative {a}", 0, valid.filter(F.col(a) < 0).count())
    for d in p["dates"]:
        record(name, f"VALID rows have {d}", 0, valid.filter(F.col(d).isNull()).count())
    record(name, "VALID rows have claim_id", 0, valid.filter(F.col("claim_id").isNull()).count())

bank = spark.table(PARTNERS["BANK"]["silver"])
record("BANK", "VALID rows have an allowed status", 0,
       bank.filter((F.col("dq_status") == "VALID") & ~F.col("status").isin(VALID_BANK_STATUS)).count())
record("BANK", "Status values are upper case", 0,
       bank.filter(F.col("status") != F.upper(F.col("status"))).count())

tpa = spark.table(PARTNERS["TPA"]["silver"])
record("TPA", "VALID rows: approved <= billed", 0,
       tpa.filter((F.col("dq_status") == "VALID") & (F.col("approved_amount") > F.col("billed_amount"))).count())
record("TPA", "VALID rows: service date not after file date", 0,
       tpa.filter((F.col("dq_status") == "VALID") & (F.col("service_date") > F.col("file_date"))).count())
record("TPA", "Provider is approved or warned", 0,
       tpa.filter(~F.col("provider_name").isin(APPROVED_PROVIDERS) &
                  ~F.array_contains("dq_warnings", "UNKNOWN_PROVIDER")).count())

repair = spark.table(PARTNERS["REPAIR"]["silver"])
record("REPAIR", "VALID rows: total = labor + parts", 0,
       repair.filter((F.col("dq_status") == "VALID") &
                     (F.col("total_amount") != F.col("labor_amount") + F.col("parts_amount"))).count())
record("REPAIR", "Shop names are trimmed", 0,
       repair.filter(F.col("shop_name") != F.trim(F.col("shop_name"))).count())

## Step 7: Known scenarios

**Why:** each messy value placed in the Day 2 files is checked by name, with the exact value Silver must produce.

In [0]:
SCENARIOS = [
    ("BANK",   "Currency text $1,250.50 converted",            "bank_ref",   "BNK202610120002",  "amount",           "1250.50"),
    ("BANK",   "Date 10/11/2026 parsed",                       "bank_ref",   "BNK202610120003",  "settlement_date",  "2026-10-11"),
    ("BANK",   "Date 11-Oct-2026 parsed",                      "bank_ref",   "BNK202610120009",  "settlement_date",  "2026-10-11"),
    ("BANK",   "Lower case status cleared standardised",       "bank_ref",   "BNK202610120003",  "status",           "CLEARED"),
    ("BANK",   "Spaces around ID removed",                     "bank_ref",   "BNK202610120004",  "status",           "CLEARED"),
    ("BANK",   "N/A claim ID flagged",                         "bank_ref",   "BNK202610120006",  "dq_status",        "INVALID"),
    ("BANK",   "Blank amount flagged",                         "bank_ref",   "BNK202610120008",  "dq_status",        "INVALID"),
    ("BANK",   "Status update keeps the latest status",        "bank_ref",   "BNK202610110003",  "status",           "RETURNED"),
    ("BANK",   "Status update counts all versions",            "bank_ref",   "BNK202610110003",  "record_versions",  "3"),
    ("BANK",   "Exact duplicate counted",                      "bank_ref",   "BNK202610120013",  "record_versions",  "2"),
    ("TPA",    "Thousands separator 4,500.00 converted",       "tpa_ref",    "TPA202610120002",  "billed_amount",    "4500.00"),
    ("TPA",    "Provider name ' MEMORIAL HERMANN ' standardised", "tpa_ref", "TPA202610120003",  "provider_name",    "Memorial Hermann"),
    ("TPA",    "Corrected duplicate keeps the valid row",      "tpa_ref",    "TPA202610120009",  "approved_amount",  "1275.00"),
    ("TPA",    "Future service date flagged",                  "tpa_ref",    "TPA202610120006",  "dq_status",        "INVALID"),
    ("TPA",    "Approved more than billed flagged",            "tpa_ref",    "TPA202610120004",  "dq_status",        "INVALID"),
    ("REPAIR", "Currency text $620.00 converted",              "invoice_no", "INV-20261012-002", "labor_amount",     "620.00"),
    ("REPAIR", "Shop name '  precision auto body' standardised", "invoice_no", "INV-20261012-008", "shop_name",     "Precision Auto Body"),
    ("REPAIR", "Missing parts amount derived",                 "invoice_no", "INV-20261012-003", "parts_amount",     "0.00"),
    ("REPAIR", "Derived parts amount still valid",             "invoice_no", "INV-20261012-003", "dq_status",        "VALID"),
    ("REPAIR", "Total mismatch flagged",                       "invoice_no", "INV-20261012-004", "dq_status",        "INVALID"),
]

for partner, test, key_col, key, col, expected in SCENARIOS:
    rows = (spark.table(PARTNERS[partner]["silver"])
            .filter(F.col(key_col) == key).select(col).collect())
    actual = rows[0][0] if rows else "row not found"
    record(partner, test, expected, actual)

## Step 8: Expected counts

**Why:** the lab data has known totals after Day 1 and Day 2. When new days of data are loaded, update these numbers or remove this step.

In [0]:
EXPECTED = {"BANK": (37, 31, 6), "TPA": (29, 24, 5), "REPAIR": (23, 21, 2)}

for name, (rows, valid, invalid) in EXPECTED.items():
    s = spark.table(PARTNERS[name]["silver"])
    record(name, "Total rows (Day 1 + Day 2)", rows, s.count())
    record(name, "VALID rows (Day 1 + Day 2)", valid, s.filter(F.col("dq_status") == "VALID").count())
    record(name, "INVALID rows (Day 1 + Day 2)", invalid, s.filter(F.col("dq_status") == "INVALID").count())

## Step 9: Test report

**Why:** one table with every test, the expected and actual values, and PASS or FAIL. Failed tests are listed first.

In [0]:
report = spark.createDataFrame(results, ["partner", "test", "expected", "actual", "result"])
display(report.orderBy(F.col("result").desc(), "partner", "test"))

failed = report.filter(F.col("result") == "FAIL").count()
print(f"Tests run: {report.count()}   Passed: {report.count() - failed}   Failed: {failed}")

## Step 10: Stop if any test failed

**Why:** when this notebook runs as part of a scheduled job, a failed test must stop the job so Gold is not built from bad Silver data.

In [0]:
if failed > 0:
    raise AssertionError(f"{failed} Silver test(s) failed. See the report above.")
print("All Silver tests passed.")